# Rudder-step 6DOF response example

このNotebookでは、`SampleGlider.stab`の線形空力モデルを使い、ラダーステップ入力に対する6自由度応答を計算する。

処理は次の順序で進む。

1. `.stab`と質量・慣性モーメントを設定する。
2. ラダーステップ応答を時間積分する。
3. 指定バンク角変化へ到達する時間と有限時間ロール応答指数を計算する。
4. 時刻歴をCSVとPNGへ保存する。

ここで使う質量・慣性モーメントは`SampleGlider`用の数値例であり、G103Aの実機MassProp値ではない。


## 1. importとリポジトリ位置


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

import pandas as pd
from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "RollRudderGain.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing "
        "src/RollRudderGain.py."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.RollRudderGain import (  # noqa: E402
    calculate_roll_response_index_by_delta_phi,
    plot_6dof_history,
    simulate_6dof_rudder_step_from_stab,
    write_6dof_history_csv,
)
from src.TrimTurnSolver import read_vspaero_stab  # noqa: E402

print("repo_root:", repo_root)


## 2. 入力モデルと解析条件

`V`と`Bref`は手入力せず、`.stab`から読み取った値を応答指数の無次元化に使う。


In [ ]:
stab_path = (
    repo_root
    / "examples"
    / "models"
    / "SampleGlider"
    / "SampleGlider.stab"
)
output_dir = (
    repo_root
    / "examples"
    / "notebooks"
    / "results"
    / "rudder_step_response"
)
output_dir.mkdir(parents=True, exist_ok=True)

if not stab_path.exists():
    raise FileNotFoundError(stab_path)

stab = read_vspaero_stab(stab_path)

mass = 100.0
inertia = {
    "Ixx": 1000.0,
    "Iyy": 75.0,
    "Izz": 1100.0,
    "Ixz": 0.0,
}

delta_r = math.radians(-10.0)
target_delta_phi = math.radians(2.0)
t_final = 5.0

display(
    pd.Series(
        {
            "stab_path": str(stab_path),
            "mass": mass,
            "Ixx": inertia["Ixx"],
            "Iyy": inertia["Iyy"],
            "Izz": inertia["Izz"],
            "Ixz": inertia["Ixz"],
            "Vinf": stab.V0,
            "Bref": stab.Bref,
            "rho": stab.rho0,
            "delta_r_deg": math.degrees(delta_r),
            "target_delta_phi_deg": math.degrees(
                target_delta_phi
            ),
            "t_final": t_final,
        },
        name="value",
    )
)


## 3. ラダーステップ応答の計算

このexampleでは推力を使わないため`trim_thrust=False`とする。エレベータは`.stab`の基準状態まわりで自動トリムする。


In [ ]:
history = simulate_6dof_rudder_step_from_stab(
    stab_path,
    mass=mass,
    Ixx=inertia["Ixx"],
    Iyy=inertia["Iyy"],
    Izz=inertia["Izz"],
    Ixz=inertia["Ixz"],
    delta_r=delta_r,
    t_final=t_final,
    delta_a=0.0,
    delta_e=None,
    trim_elevator=True,
    trim_thrust=False,
    phi0=0.0,
    theta0=None,
    psi0=0.0,
    max_step=0.02,
)

display(history.head())
print("samples:", len(history))


## 4. 有限時間ロール応答指数

指定した符号付きバンク角変化へ最初に到達する時刻を使う。

$$
K_{\phi,\delta_r}
=
\frac{\Delta\phi/\Delta t}{\delta_r}
\frac{b}{2V}
$$

`V`と`Bref`には、同じ`.stab`から読み取った`Vinf`と`Bref`を渡す。


In [ ]:
rudder_response_index = (
    calculate_roll_response_index_by_delta_phi(
        history,
        delta_r=delta_r,
        target_delta_phi=target_delta_phi,
        V=stab.V0,
        Bref=stab.Bref,
        phi0=0.0,
    )
)

display(
    pd.Series(
        {
            "reached": rudder_response_index[
                "sixdof_roll_response_reached"
            ],
            "target_delta_phi_deg": (
                rudder_response_index[
                    "sixdof_target_delta_phi_deg"
                ]
            ),
            "t_reach": rudder_response_index[
                "sixdof_t_reach"
            ],
            "phi_final_deg": math.degrees(
                rudder_response_index[
                    "sixdof_phi_final"
                ]
            ),
            "finite_time_roll_index": (
                rudder_response_index[
                    "sixdof_finite_time_roll_index"
                ]
            ),
            "reference_index_to_t_final": (
                rudder_response_index[
                    "sixdof_roll_response_index_reference"
                ]
            ),
            "error": rudder_response_index[
                "sixdof_roll_response_error"
            ],
        },
        name="value",
    )
)


## 5. 時刻歴の保存と表示


In [ ]:
csv_path = write_6dof_history_csv(
    history,
    output_dir / "rudder_step_6dof_history.csv",
)

plot_path = (
    output_dir
    / "rudder_step_6dof_history.png"
)
plot_6dof_history(
    history,
    plot_path=plot_path,
    show=True,
    degrees=True,
)

print("csv_path:", csv_path)
print("plot_path:", plot_path)


## 6. 確認事項

- `sixdof_roll_response_reached=True`
- `sixdof_roll_response_error`が空文字
- `V`と`Bref`が`.stab`の値と一致
- ラダー舵角と目標バンク角の符号が意図どおり
- 質量・慣性モーメントが対象機の単位系と一致
- 目標へ到達しない場合は、目標角、入力舵角、積分時間を物理的根拠に基づいて見直す
